## Import important libraries

In [1]:
import re
import chromadb
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.memory import ConversationBufferWindowMemory
from sentence_transformers import SentenceTransformer, util
import os
from pathlib import Path
from openai import OpenAI
from groq import Groq
import google.generativeai as genai
import gradio as gr
import json
import requests
from datetime import datetime
from typing import Literal
from pydantic import BaseModel, Field
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

dotenv_path = Path.cwd() / ".env"
load_dotenv(dotenv_path=dotenv_path, override=True)

c:\Users\hp\Desktop\All\college\Fourth_year\Second_term\GP2\GP2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\hp\AppData\Local\Temp\ipykernel_8768\3752724294.py:11: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


True

In [2]:
HF_KEY            = os.getenv("HF_TOKEN")
TELEGRAM_BOT_key  = os.getenv("TELEGRAM_BOT_TOKEN")
TELEGRAM_CHAT_ID  = os.getenv("TELEGRAM_CHAT_ID")
GITHUB_KEY        = os.getenv("GITHUB_TOKEN")
OPENAI_API_KEY    = os.getenv("OPENAI_API_KEY")
AZURE_KEY         = os.getenv("AZURE_KEY")
GROQ_API_KEY      = os.getenv("GROQ_API_KEY")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY is missing. Add it to .env and re-run the first cell.")
if not OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER_API_KEY is missing. Add it to .env and re-run.")

openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)


groq_client = Groq(api_key=GROQ_API_KEY)

In [3]:
print(OPENROUTER_API_KEY)

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


### 1. SETUP Embedding

In [4]:
import numpy as np
from huggingface_hub import InferenceClient
import time
import random

hf_client = InferenceClient(provider="hf-inference", api_key=HF_KEY)

def embed_texts_with_retry(texts: list[str], max_retries: int = 3) -> list[list[float]]:
    embeddings = []
    for text in texts:
        for attempt in range(max_retries):
            try:
                vector = np.array(hf_client.feature_extraction(text, model="BAAI/bge-m3"))
                if vector.ndim > 1:
                    vector = vector.squeeze()
                vector = vector / np.linalg.norm(vector)
                embeddings.append(vector.tolist())
                break
            except Exception as e:
                if attempt == max_retries - 1:
                    raise
                backoff = (2 ** attempt) + random.uniform(0, 0.5)
                print(f"[embed retry] attempt {attempt+1} failed: {e} — retrying in {backoff:.1f}s")
                time.sleep(backoff)
    return embeddings



### 2. SETUP ChromDB

In [5]:
DB_PATH = './chroma_db'
client = chromadb.PersistentClient(path=DB_PATH)
collection = client.get_collection(
    "ucas_knowledge_base"
    )

## 3. SEARCH WITH THRESHOLD

In [6]:
SIMILARITY_THRESHOLD = 0.4   # tune during evaluation (weeks 7-8)

def search(query: str, top_k: int = 3, metadata_filter: dict | None = None, where: dict = None, is_exact_fetch: bool = False) -> dict:
    """
    Embed query and search ChromaDB.
    metadata_filter: optional ChromaDB 'where' clause, e.g. {"program": "علم البيانات والذكاء الاصطناعي"}
    Now returns per-chunk similarity scores alongside documents/metadatas,
    so callers can rank individual chunks rather than relying on a single
    aggregate best_score.t

    is_exact_fetch: skip embedding similarity entirely and fetch ALL chunks
    matching `where` via collection.get(). Used for structured category
    lookups (study_plan / program_info / scholarship) where we want the
    whole matching set, not a top-k similarity guess.
    """
    if is_exact_fetch and where:
        print(where)
        results = collection.get(where=where)
        documents = results["documents"]
        metadatas = results["metadatas"]

        if not documents:
            print(where)
            print("  [search] No exact-category chunks matched — falling back to similarity search")

        # study_plan chunks are returned via collection.get(), which has no
        # distance metric — assign them all a fixed top score since they are
        # an exact category/program match, not a similarity match
        print(documents)
        has_answer = bool(documents)
        scores = [1.0] * len(documents)
        return {"has_answer": has_answer, "documents": documents, "metadatas": metadatas,
                "scores": scores, "best_score": 1.0}
    
    
    query_embedding = embed_texts_with_retry([query])[0]

    query_params = dict(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )
    if metadata_filter:
        query_params["where"] = metadata_filter

    results     = collection.query(**query_params)
    documents   = results["documents"][0]
    print(documents)
    metadatas   = results["metadatas"][0]
    distances   = results["distances"][0]

    # Guard: metadata filter may match no chunks at all
    if not documents:
        print("  [search] No chunks matched the filter — retrying without filter")
        query_params.pop("where", None)
        results   = collection.query(**query_params)
        documents = results["documents"][0]
        metadatas = results["metadatas"][0]
        distances = results["distances"][0]
        
    print(documents)
    if not documents:
        print("  [search] No chunks found at all")
        return {"has_answer": False, "documents": [], "metadatas": [], "scores": [], "best_score": 0.0}

    scores       = [1 - d for d in distances]      # per-chunk similarity
    best_score   = max(scores)

    print(f"  Best similarity: {best_score:.4f}  (distance: {1 - best_score:.4f})")

    if best_score < SIMILARITY_THRESHOLD:
        return {"has_answer": False, "documents": [], "metadatas": [], "scores": [], "best_score": best_score}

    return {"has_answer": True, "documents": documents, "metadatas": metadatas,
            "scores": scores, "best_score": best_score}

### 4. FALLBACK — TELEGRAM

In [ ]:
import time
import requests
from requests.exceptions import ReadTimeout, ConnectionError

TELEGRAM_BOT_TOKEN = TELEGRAM_BOT_key
# TELEGRAM_CHAT_ID already defined above

def send_fallback_telegram(student: dict, question: str) -> bool:
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
    lines = [
        "*\\[Smart Advisor\\] Unanswered Question*",
        "",
        "*Student Details*",
        f"Name  : {student.get('name',  'Not provided')}",
        f"Email : {student.get('email', 'Not provided')}",
        f"Phone : {student.get('phone', 'Not provided')}",
        f"Time  : {timestamp}",
        "",
        "*Unanswered Question*",
        f"{question}",
        "",
        "_Sent automatically by the UCAS Smart Advisor system._"
    ]
    message = "\n".join(lines)
    url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"
    payload = {
        "chat_id":    TELEGRAM_CHAT_ID,
        "text":       message,
        "parse_mode": "Markdown"
    }

    max_retries = 3
    for attempt in range(max_retries):
        try:
            resp = requests.post(url, json=payload, timeout=15)  # increased from 10
            result = resp.json()
            if result.get("ok"):
                return True
            else:
                print(f"[Telegram] API returned error: {result}")
                return False  # API error — no point retrying

        except (ReadTimeout, ConnectionError) as e:
            wait = 2 ** attempt  # 1s, 2s, 4s
            print(f"[Telegram] Attempt {attempt + 1} failed: {e}. Retrying in {wait}s...")
            if attempt < max_retries - 1:
                time.sleep(wait)
            else:
                print("[Telegram] All retries exhausted — question will be logged locally only.")
                return False

        except Exception as e:
            print(f"[Telegram] Unexpected error: {e}")
            return False

    return False

# ── TOOL FUNCTION ─────────────────────────────────────────────────────────
def record_unknown_question(question: str, name: str,
                            email: str = None, phone: str = None) -> dict:
    student = {"name": name, "email": email, "phone": phone}
    success = send_fallback_telegram(student, question)
    status = "ok" if success else "failed"
    print(f"[Fallback] Question recorded — Telegram delivery: {status}")
    
    # Optional: log to a local file as backup when Telegram fails
    if not success:
        with open("failed_questions.log", "a", encoding="utf-8") as f:
            f.write(f"\n---\nTime: {datetime.now()}\nName: {name}\nEmail: {email}\nPhone: {phone}\nQuestion: {question}\n")
    
    return {"recorded": status}

### 5. Define TOOL SCHEMA

In [8]:
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always use this tool to record any question that couldn't be answered. Also records the student's details so the advisor can follow up.",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {"type": "string", "description": "The question that couldn't be answered"},
            "name":     {"type": "string", "description": "The student's full name"},
            "email":    {"type": "string", "description": "The student's email address"},
            "phone":    {"type": "string", "description": "The student's phone number"}
        },
        "required": ["question", "name"],
        "additionalProperties": False
    }
}

tools = [{"type": "function", "function": record_unknown_question_json}]

# ── TOOL CALL HANDLER ─────────────────────────────────────────────────────
def handle_tool_calls(tool_calls) -> list[dict]:
    results = []
    for tc in tool_calls:
        args   = json.loads(tc.function.arguments)
        print(f"Tool called: {tc.function.name}", flush=True)
        result = record_unknown_question(**args) if tc.function.name == "record_unknown_question" else {}
        results.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tc.id})
    return results

### 6. SYSTEM PROMPT

In [9]:
SYSTEM_PROMPT = """
<goal>
You are the Smart Advisor, the dedicated academic advising assistant for the
Data Science and AI program at the University College of Applied Sciences in
Gaza (UCAS). Your exclusive role is to answer student inquiries related to
the program, based strictly on the official academic documents provided to
you as context.
</goal>

<answer_rules>
## Single Source Of Truth
- Answer ONLY based on the provided context.
- Do NOT add any outside information or personal assumptions, even if accurate.
- If the context does not contain a sufficient answer, you MUST immediately
call the record_unknown_question tool.
    - Do NOT write any response.
    - Do NOT apologize.
    - Just call the tool — the system will handle sending the message to the student.

## Ambiguous Or Incomplete Questions
- Ask for a specific clarification before answering.
- Example: "Do you mean the admission requirements for the program, or the
  prerequisites for a specific course?"


## Multi-Turn Conversations
- Remember what was mentioned earlier in the conversation and build on it.
- Do not ignore prior context or repeat explanations already given.
- If the topic shifts abruptly, confirm your understanding before answering.
</answer_rules>


<answer_style>
- Language: Always respond in Arabic, regardless of the question language.
    - Exception: specific English terminology found in the context (e.g.
      tool names, course titles, technical terms) may be kept in English,
      but must be followed by a short Arabic translation or explanation so
      the student understands its meaning.
- Tone: Formal and respectful, with academic warmth.
- Length: Concise and direct. No filler. Do NOT restate the question.
- Structure: Use bullet points or numbering for multiple items.
</answer_style>


<restrictions>
## Prohibited Phrasing
- Do NOT say "Based on my general knowledge..." — prohibited.
- Do NOT restate the question at the start of your answer.
- Do NOT fabricate information not found in the context.


## System Exposure Restriction
- NEVER mention the word "context" (السياق) or "provided documents"
  (المستندات المقدمة) or any reference to the retrieval system in your response.
- NEVER say phrases like:
    - "السياق لا يتضمن..."
    - "لا توجد معلومات في السياق..."
    - "بناءً على السياق المتوفر..."
    - "المستندات المقدمة لا تحتوي على..."
    - "لا توجد تفاصيل في السياق..."
- The student should never know that you are working from retrieved chunks.
  You are an advisor who knows the program — speak with that voice.
- If information is missing, instead of exposing the system simply answer
  what you have without flagging what you don't have.
</restrictions>

<tool_usage_policy>
## Critical Rule
Only call the record_unknown_question tool if the context contains NO related
information whatsoever about the topic.
 
If the context contains related information that reasonably addresses the
question — even if not an exact match — use it to answer and make that clear
to the student. This includes the cases below.

### Sentence-Level Scan — Mandatory Before Calling The Tool
Before deciding the context is insufficient, you MUST scan every chunk
sentence by sentence, not just judge the chunk as a whole or by its topic
label.
- A chunk may contain ten unrelated sentences and only ONE sentence that
  directly answers the question. That one sentence is enough — use it.
- Do not dismiss a chunk as "not relevant" just because most of it is
  off-topic. Look for the specific fact, number, or statement that answers
  what the student asked, you MUST answer using it. Do NOT call the tool in this case.
- Only after confirming that NO sentence in ANY chunk addresses the
  question should you consider calling the tool.


### Comparison Questions — Detailed Protocol
When a student asks to compare two or more programs, tracks, or courses,
provide a detailed and informative comparison that covers every relevant
dimension found in the context, such as:
    - Focus
    - Content
    - Goals
    - Practical vs. theoretical nature
    - Similarities and differences
 
- Do not invent differences. If two options share a trait, say so explicitly.
- If the student's question implies a personal fit decision ("which is
  better for me?").

### General Questions
- If the context contains program overviews, course descriptions, or study
  plan details, use them to answer questions about what a program covers,
  who it is for, or how it differs from others and any other queries related to that context.


### Examples Of When You MUST Answer (Not Call The Tool)
- Student asks about DS scholarships → context has general college
  scholarships → ANSWER using the context, note they apply to all programs
  including DS.
- Student asks "ما الفرق بين علم البيانات والأمن السيبراني؟" → context has
  program_info chunks for both → ANSWER by summarizing each program's focus,
  degree, duration, and admission requirements. This is a valid complete answer.


### When To Escalate
Only escalate when the topic is completely absent from the context:
- A question about a university not mentioned anywhere in the context.
- A question about a specific policy or number that no chunk contains.
- A question whose topic has zero overlap with any retrieved chunk.
</tool_usage_policy>

<response_length>
- For simple factual questions (what is X, who teaches X, when is X):
  Answer in 5-10 lines maximum. List the facts directly.
- Do NOT add career advice, encouragement, or elaboration unless the student
  explicitly asks for it.
- Only expand with personalization when the student asks something like
  "هل يناسبني هذا المساق؟" or "ما رأيك في هذا التخصص؟"
</response_length>

<anti_hallucination_rules>
## Rule 1 — The "Can I Point To It?" Test
Before writing any sentence, ask yourself: "Does this exact information
appear in the provided context chunks?"
- If YES → write it.
- If NO → do not write it, even if you are certain it is true.
This applies to: names, numbers, tools, languages, policies, dates,
requirements, descriptions, comparisons, outcomes — everything.
 
## Rule 2 — No Gap-Filling
- If the context covers a topic partially, answer only the part covered.
- NEVER fill gaps with your general knowledge, even if you are confident.

## Rule 3 — No Inference
Do not draw conclusions that are not explicitly stated in the context.
- Wrong: context says "يعتمد على الرياضيات" → you write "إذن تحتاج إلى Python"
- Right: only state what the context says word-for-word.

## Rule 3.1 — Eligibility And Admission Questions Are Hard Boundaries
Questions about whether a student CAN or CANNOT enroll in a program (based
on academic track, GPA, or any admission requirement) must be treated as
strict factual lookups, not as something to soften or work around.
- If the context states the admission requirement (e.g. specific allowed
  tracks, minimum GPA) and the student's stated track or GPA does not meet
  it, you MUST state plainly that they do not meet the requirement as
  listed.
- Do NOT invent alternative pathways, conditions, extra courses, bridging
  programs, exceptions, or workarounds that would allow an ineligible
  student to join — UNLESS such a pathway is explicitly stated in the
  context. If no such pathway appears in the context, none exists for the
  purposes of your answer.
- Do NOT soften an eligibility rejection by suggesting the student "could
  still join if they take extra courses" or similar, unless that exact
  statement appears in the context.
- This rule applies even within a long conversation. If you already
  correctly stated a student is ineligible earlier, do not contradict
  yourself later in the same conversation by fabricating a workaround.
- An eligibility fact is exactly as final and unchangeable as any other
  fact in the context — do not treat it as negotiable or as something you
  can be more "helpful" about by inventing exceptions.
- Do NOT speculate about the possible existence of exceptions, special
  tracks, or additional requirements that "might" allow an ineligible
  student to join, even while admitting you are unsure or recommending
  they ask elsewhere. Suggesting an unconfirmed possibility is still
  fabrication — only state what the context confirms.
- Do NOT tell the student to contact the university, an advisor outside
  this conversation, or any other party to "ask about exceptions." If the
  context does not mention an exception, the correct answer is simply that
  they do not meet the listed requirement — full stop. (If your own
  system separately wants to offer escalation to a human advisor, that is
  handled through the record_unknown_question tool, not through your own
  suggestion to "go ask someone.")
### Example
- Wrong: "هذا لا يلبي الشرط حسب الشروط المعلنة. يُنصح بالتواصل مع الجامعة
  للاستفسار عن إمكانية وجود مسارات استثنائية قد تسمح له بالانضمام."
- Right: "لا يستوفي الطالب من الفرع الأدبي شرط القبول في هذا التخصص، حيث
  يُشترط أن يكون الطالب من الفرع العلمي أو الصناعي أو تكنولوجيا المعلومات
  بمعدل 70% فأعلى.
  
## Rule 4 — Hallucination Signal Words
Never use these words — they signal you are drawing from general knowledge,
not the context:
    - "عادةً"
    - "في الغالب"
    - "بشكل عام"
    - "من المتعارف عليه"
    - "من المعروف أن"
    - "يُعدّ من"
    - "في مجال X يُستخدم عادةً"
If you find yourself writing them, stop and delete that sentence.
 
## Rule 5 — Matching Student Profile To Programs Or Courses
- Present programs that match the student's interests and GPA positively,
  describing what each offers and how it aligns with their profile.
- Do NOT use negative framing. Never say a program is "غير مناسب"،
  "لا يناسبك"، "لا نوصي به"، or any equivalent.
- If multiple programs fit the student's profile, present each one's
  strengths relevant to their interests — let the student decide.
- If one program clearly aligns better, you may highlight that alignment
  without dismissing the others.
### Example Of WRONG Framing
"علم البيانات مناسب لك، أما الأمن السيبراني فلا يتوافق مع اهتماماتك."
 
### Example Of CORRECT Framing
"بناءً على اهتمامك بالذكاء الاصطناعي وشغفك بالرياضيات، يوفر تخصص علم
البيانات والذكاء الاصطناعي مساقات في تعلم الآلة ورؤية الحاسوب ومعالجة اللغة
الطبيعية. كما يوفر تخصص هندسة أمن المعلومات السيبراني تخصصًا في حماية
الأنظمة والشبكات إذا كان هذا المجال يثير اهتمامك."
</anti_hallucination_rules>

<planning_guidance>
When drafting a response:
1. Check whether the context contains relevant information for the question.
2. If absent entirely, call record_unknown_question immediately — no text response.
3. If present (fully or partially), apply the anti_hallucination_rules to every sentence.
4. Apply the tool_usage_policy for comparison vs. general questions.
5. Apply response_length rules to decide how much detail to include.
6. Apply restrictions to ensure no system/context exposure and no cross-university suggestions.
7. Format the final answer per answer_style.
</planning_guidance>


<output>
- Respond only in Arabic, regardless of the question's language. Specific
  English terminology that appears in the context (e.g. tool names, course
  titles, technical terms) may be kept in English, but must be followed by a
  brief Arabic translation or explanation so the student understands its meaning.
- Speak as an advisor with direct knowledge of the program — never reveal
  that answers come from retrieved documents or a knowledge base.
- Keep answers concise unless the student asks for elaboration or personalization.
- Never include hallucination signal words or unverified claims.
</output>
"""

### 7. CONTEXT BUILDER

In [10]:
def build_context(documents: list[str]) -> str:
    context_parts = []
    for i, doc in enumerate(documents):
        context_parts.append(f"[{i+1}] {doc}")
        print(i, doc)
    return "\n\n".join(context_parts)

### 8. LLM CLIENTS & GENERATION HELPERS

In [11]:
github_client = OpenAI(base_url="https://models.github.ai/inference", api_key=GITHUB_KEY)

def generate_gpt4o(query: str, context: str) -> str:
    r = github_client.chat.completions.create(
        model="openai/gpt-4o", messages=_make_messages(query, context),
        temperature=1.0, top_p=0.8, max_tokens=1000
    )
    return r.choices[0].message.content

### 9. STUDENT PROFILE (Pydantic)

In [12]:
# Stores structured information about the student, collected during onboarding.
# Used to personalise query rewriting and LLM answers.

class StudentProfile(BaseModel):
    # Admission eligibility
    gpa: float | None = Field(None, description="Tawjihi GPA as a percentage (0-100)", ge=0, le=100)
    academic_track: Literal["علمي", "صناعي", "تكنولوجيا معلومات", "أدبي", "تجاري", "أخرى"] | None = Field(
        None, description="High school academic track"
    )
    enrollment_status: Literal["طالب حالي", "طالب جديد", "مهتم بالالتحاق"] | None = Field(
        None, description="Whether the student is currently enrolled or prospective"
    )
    # Interests
    likes_math: bool | None = Field(None, description="Enjoys mathematics and statistics")
    has_programming_background: bool | None = Field(None, description="Has prior programming experience")
    interest_areas: list[Literal[
        "علم البيانات", "الذكاء الاصطناعي", "أمن المعلومات",
        "شبكات الحاسوب", "هندسة الحاسوب", "غير محدد"
    ]] = Field(default_factory=list, description="Areas of academic or professional interest")
    # Academic goals
    degree_preference: Literal["بكالوريوس", "دبلوم", "غير محدد"] | None = Field(
        None, description="Preferred degree level"
    )
    needs_financial_aid: bool | None = Field(None, description="Interested in scholarships")

    # Priority fields that drive onboarding — ordered by importance
    _PRIORITY = ["gpa", "academic_track", "likes_math", "interest_areas", "degree_preference"]

    def missing_fields(self) -> list[str]:
        """Return priority fields still None (or empty list for interest_areas)."""
        result = []
        for f in self._PRIORITY:
            v = getattr(self, f)
            if v is None or (isinstance(v, list) and len(v) == 0):
                result.append(f)
        return result

    def is_complete(self) -> bool:
        return len(self.missing_fields()) == 0

    def to_context_string(self) -> str:
        """Arabic summary injected into prompts."""
        parts = []
        if self.gpa is not None:
            parts.append(f"المعدل في الثانوية: {self.gpa}%")
        if self.academic_track:
            parts.append(f"الفرع الدراسي: {self.academic_track}")
        if self.enrollment_status:
            parts.append(f"الحالة: {self.enrollment_status}")
        if self.likes_math is not None:
            parts.append("يستمتع بالرياضيات" if self.likes_math else "لا يفضل الرياضيات")
        if self.has_programming_background is not None:
            parts.append("لديه خلفية برمجية" if self.has_programming_background else "لا توجد خلفية برمجية")
        if self.interest_areas:
            parts.append(f"اهتماماته: {', '.join(self.interest_areas)}")
        if self.degree_preference:
            parts.append(f"يريد: {self.degree_preference}")
        if self.needs_financial_aid:
            parts.append("مهتم بالمنح الدراسية")
        return " | ".join(parts) if parts else "لا توجد معلومات بعد"

### 10. PROFILE EXTRACTION LLM

In [13]:
# A dedicated LLM call that reads the student's message and updates the profile.
# Temperature = 0.0 for deterministic extraction.

_EXTRACTION_SYSTEM = """
You are an assistant who extracts student information from a conversation message.
Return ONLY JSON object with these keys (if the information is not exist use null):
{
  \\"gpa\\": floating number from 0 to 100 or null,
  \\"academic_track\\": one of [علمي، صناعي، تكنولوجيا معلومات، أدبي، تجاري، أخرى] or null,
  \\"enrollment_status\\": one of [طالب حالي، طالب جديد، مهتم بالالتحاق] or null,
  \\"likes_math\\": null or true/false,
  \\"has_programming_background\\": null or true/false,
  \\"interest_areas\\": one of [علم البيانات، الذكاء الاصطناعي، أمن المعلومات، شبكات الحاسوب، هندسة الحاسوب، غير محدد] or null,
  \\"degree_preference\\": one of [بكالوريوس، دبلوم، غير محدد] or null,
}
Rules: 
- Only extract a field if the student's message is actually answering the question about THAT field,
  or explicitly volunteers that information.

- The student's message may be a short answer to a question the assistant just asked.
  Use the conversation context to determine which field this answer applies to.

- For yes/no fields (likes_math): use your full language understanding to determine if the student
  is confirming or denying — this includes any affirmative or negative expression in Arabic, English,
  or dialect (formal, colloquial, Gaza dialect, or any other variation). You do not need an explicit
  list — trust your judgment. Examples of affirmatives (not exhaustive): نعم، أجل، آه، أيوة، صح،
  تمام، أكيد، بالتأكيد، yes، yeah، sure، طبعاً، ماشي، يلا. Examples of negatives (not exhaustive):
  لا، مش، لأ، no، nope، أبداً، مو، ما.

- For degree_preference: map any duration expression to the correct value —
  "سنتان"، "سنتين"، "2"، "two years" → دبلوم;
  "أربع سنوات"، "4"، "أربعة"، "four years" → بكالوريوس.

- Do not invent information not in the message or implied by the immediate question context.

- Do not modify fields unless the student explicitly corrects information.

- Return only JSON, without explanation or markdown.
"""

def extract_profile(user_message: str, current: StudentProfile) -> StudentProfile:
    """Call LLM to extract profile fields from student message and merge with current profile."""
    try:
        resp = groq_client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[
                {"role": "system", "content": _EXTRACTION_SYSTEM},
                {"role": "user",   "content": (
                    f"الملف الحالي: {current.model_dump_json()}\\n\\n"
                    f"رسالة الطالب: {user_message}"
                )}
            ],
            temperature=0.0,
            max_tokens=400
        )
        raw = re.sub(r"```json|```", "", resp.choices[0].message.content).strip()
        extracted = json.loads(raw)

        # ── Strip whitespace from all extracted string values ──────────────
        cleaned = {}
        for key, value in extracted.items():
            if isinstance(value, str):
                cleaned[key] = value.strip()
            elif isinstance(value, list):
                cleaned[key] = [v.strip() if isinstance(v, str) else v for v in value]
            else:
                cleaned[key] = value

        # Merge: only fill None / empty-list fields from extraction
        current_data = current.model_dump()
        for key, value in cleaned.items():
            if value is None:
                continue
            existing = current_data.get(key)
            if existing is None or (isinstance(existing, list) and len(existing) == 0):
                current_data[key] = value

        return StudentProfile(**current_data)
    except Exception as e:
        print(f"[Profile extraction error] {e}")
        return current   # return unchanged on any failure

### 11. ONBOARDING QUESTION SEQUENCER

In [14]:
# Maps missing profile fields → natural Arabic questions.
# No LLM needed — pure lookup table for speed and reliability.

_FIELD_QUESTIONS: dict[str, str] = {
    "gpa": (
        "للبدء، ما معدلك في الثانوية العامة (التوجيهي)؟ "
        "هذا يساعدني في معرفة البرامج التي تؤهل للقبول."
    ),
    "academic_track": (
        "ما فرعك الدراسي في الثانوية؟ "
        "(علمي / صناعي / تكنولوجيا معلومات / أدبي / تجاري / أخرى)"
    ),
    "likes_math": (
        "هل تستمتع بالرياضيات والإحصاء؟ "
        "أسألك لأن بعض التخصصات كعلم البيانات تعتمد عليهما بشكل كبير."
    ),
    "interest_areas": (
        "ما الذي يثير اهتمامك أكثر؟ "
        "(تحليل البيانات / الذكاء الاصطناعي / أمن المعلومات / الشبكات / هندسة الحاسوب)"
    ),
    "degree_preference": (
        "هل تفضل الحصول على درجة البكالوريوس (4 سنوات) أم الدبلوم (سنتان)؟"
    ),
}

def next_onboarding_question(profile: StudentProfile) -> str | None:
    """Return the next question to ask, or None if profile is complete."""
    for field in profile.missing_fields():
        q = _FIELD_QUESTIONS.get(field)
        if q:
            return q
    return None

### 12. METADATA FILTER — LLM-BASED PROGRAM/COURSE DETECTION

In [15]:
# Extracts the program or course name mentioned in the query (if any)
# and returns a ChromaDB 'where' filter to restrict retrieval.
#
# Known program names in the DB (must match metadata exactly):
KNOWN_PROGRAMS = [
    "علم البيانات والذكاء الاصطناعي",
    "هندسة أمن المعلومات السيبراني",
    "شبكات الحاسوب والإنترنت",
    "صيانة الأجهزة الذكية",
    "أمن المعلومات"
]

KNOWN_COURSES = [
    "قرآن كريم",
    "اللغة الإنجليزية",
    "أساسيات علم البيانات والذكاء الاصطناعي",
    "لغة برمجة (عملي)",
    "مقدمة في الحوسبة (عملي)",
    "لغة برمجة",
    "مقدمة في الحوسبة",
    "تفاضل وتكامل 1",
    "تفاضل وتكامل 2",
    "دراسات في السيرة",
    "لغة إنجليزية تخصصية",
    "تراكيب بيانات",
    "لغات برمجة علم البيانات (عملي)",
    "الرياضيات المنفصلة",
    "لغات برمجة علم البيانات",
    "مبادئ الإحصاء والاحتمالات",
    "الجبر الخطي",
    "مقدمة في قواعد البيانات (عملي)",
    "برمجة الذكاء الاصطناعي (عملي)",
    "تحليل وتمثيل البيانات",
    "مقدمة في قواعد البيانات",
    "برمجة للذكاء الاصطناعي",
    "تحليل وتمثيل البيانات (عملي)",
    "التنقيب عن البيانات (عملي)",
    "معمارية الحاسوب",
    "مبادئ الشبكات",
    "برمجة تعلم الآلة",
    "تصميم وتحليل الخوارزميات",
    "التنقيب عن البيانات",
    "برمجة تعلم الآلة (عملي)",
    "اللغة العربية",
    "النظم المغموسة (عملي)",
    "نظم التشغيل",
    "إنترنت الأشياء",
    "الحوسبة السحابية",
    "تمييز الأنماط",
    "النظم المغموسة",
    "العمل الحر",
    "التعلم العميق (عملي)",
    "مناهج البحث العلمي والكتابة العلمية",
    "مخازن البيانات",
    "معالجة الصور الرقمية",
    "التعلم العميق",
    "الأنظمة الخبيرة",
    "ريادة الأعمال",
    "معالجة اللغات الطبيعية (عملي)",
    "أخلاقيات الذكاء الاصطناعي",
    "برمجة الروبوت (التعليم المعزز)",
    "برمجة الروبوت (التعليم المعزز) عملي",
    "عمليات تعلم الآلة (MLOps)",
    "معالجة اللغات الطبيعية",
    "مشروع تخرج (2)",
    "متطلب تخصص اختياري",
    "دراسات في العقيدة",
    "هندسة برمجيات",
    "التدريب الميداني",
    "البيانات الكبيرة",
    "استرجاع المعلومات",
    "علم الإدراك والمعرفة",
    "برمجة الروبوت"
]

# Programs that share a generic field name but differ in degree level/duration.
# If the student uses the generic name without disambiguating, search both.
AMBIGUOUS_PROGRAM_GROUPS: list[set[str]] = [
    {"هندسة أمن المعلومات السيبراني", "أمن المعلومات"},
]

_DEGREE_DISAMBIGUATION_KEYWORDS = [
    "بكالوريوس", "دبلوم", "سنتين", "سنتان", "أربع سنوات", "اربع سنوات",
    "هندسة أمن", "سيبراني",
]

def _expand_ambiguous_programs(programs: list[str], query: str) -> list[str]:
    """Safety net on top of the LLM extraction: if an ambiguous field name
    was matched and the student didn't specify degree/duration, expand to
    the full ambiguous group so both programs get searched."""
    if any(k in query for k in _DEGREE_DISAMBIGUATION_KEYWORDS):
        return programs
    result = set(programs)
    for group in AMBIGUOUS_PROGRAM_GROUPS:
        if result & group:
            result |= group
    return list(result)

# ── Maps an LLM-detected boolean flag to its Chroma "category" value. ──────
# Adding a new filterable category in the future = adding ONE line here plus
# one clause in the prompt below. Nothing else in this file needs to change.
CATEGORY_FLAG_MAP = {
    "is_study_plan": "study_plan",
    "is_program_info": "program_info",
    "is_scholarship": "scholarship",
    "is_career_opportunities": "career_opportunities",
}
 
 
def _related_faq_filter(category: str) -> dict:
    """
    FAQ paragraphs are sometimes hand-tagged as relevant to a specific
    category even though their own category is "faq" (e.g. a paragraph
    discussing course load lives under category=faq but carries
    "study_plan_related": true). Whenever a category filter fires, we
    ALSO pull in any faq chunk carrying "<category>_related": true.
 
    If that flag doesn't exist yet for a given category (e.g. nobody has
    tagged any FAQ paragraph as "career_opportunities_related" yet), this
    where-clause simply matches zero chunks — safe no-op, no crash.
    """
    return {"$and": [{"category": "faq"}, {f"{category}_related": True}]}


_FILTER_SYSTEM = f"""
You are an assistant who determines whether a student's question refers to a specific academic program or course, and what category of information they want.

List of available programs:
{chr(10).join(f'- {p}' for p in KNOWN_PROGRAMS)}

Your task:
1. Extract ALL programs mentioned or clearly referred to. Return their exact names from the list {KNOWN_PROGRAMS}.
2. Extract ALL course codes mentioned (e.g. DSAI1301). Return them as a list.
3. Extract ALL course names mentioned or clearly referred to. Return their exact names from the list {KNOWN_COURSES}.
4. If no specific program is mentioned AND the question is not a general/comparison question about "all majors" (see rule 9), return "علم البيانات والذكاء الاصطناعي".
5. If no specific course is mentioned, return null.
6. Set "is_study_plan" to true if the student is asking about the full study plan,
   curriculum, semester structure, OR the set of courses/subjects a program offers
   — including comparisons like "ما الفرق بين مساقات X ومساقات Y؟" or
   "ما هي المساقات في تخصص كذا؟". This is broader than just literal "خطة دراسية" wording.
7. Determine if the student is asking about general program facts — admission requirement/GPA cutoff, degree type, duration, credit hours, college/department, or a general overview of the program (but NOT the full semester-by-semester curriculum) or something like that. If so, set "is_program_info" to true.
8. Determine if the student is asking about scholarships, financial aid, grants, or fee discounts something like: منحة، منح، مساعدة مالية، إعفاء. or any other similar words. If so, set "is_scholarship" to true.
9. Determine if the student is asking a broad question about ALL specializations at UCAS, OR comparing a program to "other majors" / "بقية التخصصات" / "باقي البرامج" without naming those other majors specifically. If so, set "all_programs" to true and include ALL programs from the list in "programs".
10. Set "is_career_opportunities" to true if the student is asking about jobs, career paths,
   salaries, the job market, remote work, freelancing, or life after graduation

IMPORTANT — a question can need MORE THAN ONE of flags 6-9 at the same time.
Example: "ما هي خطة الدراسة وما هي فرص العمل بعد التخرج؟" → is_study_plan=true AND
is_career_opportunities=true, BOTH at once. Never suppress one flag because another fired.

CRITICAL RULE — Ambiguous Program Names:
=========================================
"أمن المعلومات" / "الأمن السيبراني" / "أمن سيبراني" is ambiguous — it can mean
EITHER of TWO different programs:
  - "هندسة أمن المعلومات السيبراني" (بكالوريوس - أربع سنوات)
  - "أمن المعلومات" (دبلوم - سنتان)
If the student's message does not clearly specify which one (no mention of
"بكالوريوس", "دبلوم", "أربع سنوات", "سنتين", or "هندسة"), extract BOTH program
names into "programs". Only extract one if the student disambiguates
(e.g. "دبلوم" → diploma only, "بكالوريوس"/"هندسة" → the engineering bachelor's only).

CRITICAL RULES for course name matching:
==========================================
- Only extract a course name if the student is EXPLICITLY asking about that 
  specific course by name or code.
- Do NOT match a course just because the query contains a word that appears 
  in a course name.

Examples of when NOT to match a course:
- "لماذا الرياضيات مهمة؟" → do NOT match "الرياضيات المنفصلة" 
  (student is asking about math in general, not that specific course)
- "هل البرمجة صعبة؟" → do NOT match "لغة برمجة" 
  (student is asking generally, not about that course)
- "ما أهمية الذكاء الاصطناعي؟" → do NOT match "أساسيات علم البيانات والذكاء الاصطناعي"

Examples of when TO match a course:
- "ما محتوى مساق الرياضيات المنفصلة؟" → match "الرياضيات المنفصلة" ✓
- "من يدرّس DSAI1308؟" → match by course code ✓
- "ما متطلبات مساق تراكيب البيانات؟" → match "تراكيب بيانات" ✓

The student must be asking ABOUT the course itself, not just mentioning 
a topic that relates to it.

Examples for the category flags:
- "كم عدد الساعات المعتمدة لتخصص علم البيانات؟" → is_program_info=true, programs=["علم البيانات والذكاء الاصطناعي"]
- "ما هو معدل القبول لتخصص أمن المعلومات؟" → is_program_info=true, programs=["هندسة أمن المعلومات السيبراني", "أمن المعلومات"]
- "ما هي خطة دراسة تخصص أمن المعلومات؟" → is_study_plan=true, programs=["هندسة أمن المعلومات السيبراني", "أمن المعلومات"]
- "هل يوجد منح دراسية؟" → is_scholarship=true
- "ما شروط منحة ويبقى الأمل؟" → is_scholarship=true
- "ما الفرق بين علم البيانات وباقي التخصصات في الجامعة؟" → all_programs=true, programs=[all programs]
- "ما هي التخصصات المتاحة في UCAS؟" → all_programs=true, programs=[all programs]
- "ما الفرق بين علم البيانات والأمن السيبراني؟" → all_programs=false, programs=["علم البيانات والذكاء الاصطناعي", "هندسة أمن المعلومات السيبراني"]

Return ONLY JSON:
{{"programs": [...] or ["علم البيانات والذكاء الاصطناعي"], "course_codes": [...] or null, "course_names": [...] or null, "is_study_plan": true or false, "is_program_info": true or false, "is_scholarship": true or false, "is_career_opportunities": true or false, "all_programs": true or false}}
"""

def _category_and_programs_filter(category: str, programs: list[str]) -> dict:
    prog_f = {"program": {"$in": programs}} if len(programs) > 1 else {"program": programs[0]}
    return {"$and": [{"category": category}, prog_f]}

def detect_metadata_filter(query: str, memory: "ConversationBufferWindowMemory | None" = None) -> dict | None:
    """
    Returns (filters, programs, course_codes, course_names) where `filters`
    is a LIST of independent exact-fetch requests:
        [{"category": "study_plan", "where": {...}}, {"category": "career_opportunities", "where": {...}}, ...]
 
    Unlike the old version, this NEVER stops at the first matching category —
    every flag the LLM set gets its own filter entry, so a compound question
    ("خطة الدراسة وفرص العمل") returns filters for BOTH categories.
 
    course_codes/course_names/programs are still returned separately for the
    similarity-search fallback path, exactly as before.
    """

    try:
        resp = openrouter_client.chat.completions.create(
            model="openai/gpt-4o-mini",
            messages=[
                {"role": "system", "content": _FILTER_SYSTEM},
                {"role": "user",   "content": query}
            ],
            temperature=0.0,
            max_tokens=200
        )
        raw = re.sub(r"```json|```", "", resp.choices[0].message.content).strip()
        print(f"[detect_metadata_filter raw] {raw}")
        detected = json.loads(raw)

        raw_programs = detected.get('programs') or []
        if isinstance(raw_programs, str):
            raw_programs = [raw_programs]  # wrap stray string in a list
        programs = [p for p in raw_programs if p in KNOWN_PROGRAMS]

        all_programs_flag = detected.get("all_programs", False)
        if all_programs_flag:
            programs = list(KNOWN_PROGRAMS)

        # Safety net for the أمن المعلومات ambiguity, independent of the LLM call
        programs = _expand_ambiguous_programs(programs, query)
        print(f"[Filter] programs={programs}")
        
        course_codes = detected.get("course_codes") or []
        print(f"[Filter] course_codes={course_codes}")

        raw_course_names = detected.get("course_names") or []
        if isinstance(raw_course_names, str):
            raw_course_names = [raw_course_names]  # wrap stray string in a list
        course_names     = [p for p in raw_course_names if p in KNOWN_COURSES]
        print(f"[Filter] course_names={course_names}")

        for flag in CATEGORY_FLAG_MAP:
                print(f"  {flag}={detected.get(flag, False)}")
    
        filters = []
    
        # ── Broad "all programs" comparison — its own independent filter ──
        if all_programs_flag and not detected.get("is_study_plan", False):
            filters.append({
                "category": "comparison",
                "where": {"$and": [
                    {"category": {"$in": ["program_info", "career_opportunities"]}},
                    {"program": {"$in": programs}}
                ]}
            })
    
        # ── Every other category flag gets its OWN filter entry. ──────────
        # Scholarship isn't program-scoped (college-wide), everything else is.
        if detected.get("is_scholarship", False):
            filters.append({"category": "scholarship", "where": {"category": "scholarship"}})
    
        for flag, category in CATEGORY_FLAG_MAP.items():
            if category == "scholarship":
                continue  # handled above (no program scoping)
            if detected.get(flag, False):
                p = programs or ["علم البيانات والذكاء الاصطناعي"]
                filters.append({
                    "category": category,
                    "where": _category_and_programs_filter(category, p)
                })
    
        return filters, programs, course_codes, course_names
    
    except Exception as e:
        print(f"[Filter detection error] {e}")
        return [], [], [], []
 
 

### 13. MULTI-QUERY SEARCH

In [16]:
def generate_queries(user_message: str, memory: ConversationBufferWindowMemory, profile: StudentProfile, n: int = 2) -> list[str]:
    chat_history = memory.load_memory_variables({})["history"]
    history_msgs = [
        {"role": ("user" if m.type == "human" else "assistant"), "content": m.content}
        for m in chat_history
    ]
    profile_ctx = profile.to_context_string()
    prompt = [
        {"role": "system", "content": f"""You are a search optimisation assistant for an academic knowledge base.
Student profile: {profile_ctx}
Given the student's question, generate {n} search queries approaching the topic from completely different angles.
Each query must:
- Use different keywords than the others
- Target a different aspect of the topic
- IMPORTANT: If the question contains pronouns or references (e.g. 'خطته', 'هذا البرنامج', 'نفس التخصص'),
  resolve them using the conversation history before generating queries.
  Never generate queries with unresolved pronouns.
- Use the student profile to be specific where helpful
- Avoid mere synonyms — think about what section headings or labels appear in the document

You MUST respond with ONLY a raw JSON array of {n} strings, no markdown, no backticks, no explanation.
Correct format: ["query 1", "query 2", "query 3"]"""
        },
        *history_msgs,
        {"role": "user", "content": user_message}
    ]
    try:
        resp = github_client.chat.completions.create(
            model="gpt-4o-mini", messages=prompt, temperature=0.7, max_tokens=300
        )
        raw = resp.choices[0].message.content.strip()
        raw = re.sub(r"```json|```", "", raw).strip()  # strip markdown if present
        print(f"[generate_queries raw] {raw}")          # ← see exactly what came back
        queries = json.loads(raw)
        all_queries = [user_message] + queries[:n]
        return all_queries
    except Exception as e:
        print(f"[generate_queries error] {e}")
        return [user_message]

def estimate_tokens(text: str) -> int:
    """
    Rough token estimate without a tokenizer dependency.
    Arabic text tends to run ~2-3 chars/token with most LLM tokenizers;
    using 2.5 as a conservative middle ground. English/code text in the
    same chunk (course codes, instructor emails) skews this slightly, so
    err on the side of overestimating rather than under.
    """
    return max(1, int(len(text) / 2.2))

def _round_robin_trim(buckets: "dict[str, list[tuple[str, dict]]]", max_context_tokens: int):
    """
    Given {category: [(doc, meta), ...]}, returns (documents, metadatas,
    running_tokens) built by taking one chunk from each non-empty category
    in turn, stopping once the token budget is hit — but always giving
    every category at least ONE chunk first, even if that alone exceeds
    budget. This is what keeps a compound query (e.g. scholarship +
    career_opportunities, ~3,600 tokens combined if dumped in full) from
    silently starving one category or blowing past a small model's total
    request limit (e.g. GitHub Models' 8,000-token cap on gpt-4o-mini).
    """
    buckets = {c: list(chunks) for c, chunks in buckets.items() if chunks}
    merged_docs: list[str] = []
    merged_meta: list[dict] = []
    running_tokens = 0
    guaranteed = set()  # categories that already got their first chunk
 
    categories = list(buckets.keys())
    round_idx = 0
    while categories:
        cat = categories[round_idx % len(categories)]
        doc, meta = buckets[cat].pop(0)
        doc_tokens = estimate_tokens(doc)
 
        must_keep = cat not in guaranteed  # first chunk per category is free
        if not must_keep and running_tokens + doc_tokens > max_context_tokens and merged_docs:
            if not buckets[cat]:
                categories.remove(cat)
                continue
            round_idx = (round_idx + 1) % len(categories)
            continue
 
        merged_docs.append(doc)
        merged_meta.append(meta)
        running_tokens += doc_tokens
        guaranteed.add(cat)
 
        if not buckets[cat]:
            categories.remove(cat)
        if categories:
            round_idx = (round_idx + 1) % len(categories)
 
        if running_tokens >= max_context_tokens and len(guaranteed) == len(buckets):
            break
 
    return merged_docs, merged_meta, running_tokens
    
from concurrent.futures import ThreadPoolExecutor, as_completed 

def multi_query_search(user_message: str, memory: "ConversationBufferWindowMemory",
                        profile: "StudentProfile", top_k: int = 3,
                        max_chunks: int = 6, max_context_tokens: int = 3000) -> dict:
    """
    1. Detect ALL active category filters from the query (not just one).
    2. Exact-fetch every active filter, PLUS its related FAQ paragraphs
       (via "<category>_related": true), and merge everything together.
    3. Only if NONE of the exact filters found anything, fall back to the
       existing multi-query similarity search (unchanged from before).
    """
    filters, programs, course_codes, course_names = detect_metadata_filter(user_message, memory)
 
    if filters:
        # Per-category buckets (not one flat list) so we can round-robin
        # across categories below — a compound query must not let one big
        # category (e.g. career_opportunities) starve a smaller one
        # (e.g. scholarship) out of the token budget entirely.
        seen_keys: set[str] = set()
        buckets: dict[str, list[tuple[str, dict]]] = {}
 
        for f in filters:
            bucket = buckets.setdefault(f["category"], [])
 
            result = search(user_message, where=f["where"], is_exact_fetch=True)
            if result["has_answer"]:
                for doc, meta in zip(result["documents"], result["metadatas"]):
                    key = re.sub(r"\s+", "", doc)[:200]
                    if key not in seen_keys:
                        seen_keys.add(key)
                        bucket.append((doc, meta))
 
            related = search(user_message, where=_related_faq_filter(f["category"]), is_exact_fetch=True)
            if related["has_answer"]:
                for doc, meta in zip(related["documents"], related["metadatas"]):
                    key = re.sub(r"\s+", "", doc)[:200]
                    if key not in seen_keys:
                        seen_keys.add(key)
                        bucket.append((doc, meta))
 
        if any(buckets.values()):
            merged_docs, merged_meta, running_tokens = _round_robin_trim(buckets, max_context_tokens)
            print(f"[Multi-Query] {len(filters)} active filter(s) -> {len(merged_docs)} chunks kept "
                  f"across {len(buckets)} categories (~{running_tokens} est. tokens, budget={max_context_tokens})")
            return {"has_answer": True, "documents": merged_docs, "metadatas": merged_meta, "best_score": 1.0}
 
        print("  [Multi-Query] none of the exact-category filters matched anything — falling back to similarity search")
 
    # ── Similarity-search fallback: identical to the previous implementation ──
    metadata_filter = None
    if course_codes:
        metadata_filter = {"course_code": {"$in": course_codes}} if len(course_codes) > 1 else {"course_code": course_codes[0]}
        if programs:
            prog_f = {"program": {"$in": programs}} if len(programs) > 1 else {"program": programs[0]}
            metadata_filter = {"$and": [metadata_filter, prog_f]}
    elif course_names:
        metadata_filter = {"course_name": {"$in": course_names}} if len(course_names) > 1 else {"course_name": course_names[0]}
        if programs:
            prog_f = {"program": {"$in": programs}} if len(programs) > 1 else {"program": programs[0]}
            metadata_filter = {"$and": [metadata_filter, prog_f]}
    elif programs:
        metadata_filter = {"program": {"$in": programs}} if len(programs) > 1 else {"program": programs[0]}
 
    queries = generate_queries(user_message, memory, profile)
    print(f"[Multi-Query] {len(queries)} queries: {queries}")
 
    chunk_map: dict[str, tuple[float, str, dict]] = {}
    best_score = 0.0
 
    with ThreadPoolExecutor(max_workers=len(queries)) as executor:
        futures = {executor.submit(search, q, top_k, metadata_filter): q for q in queries}
        for future in as_completed(futures):
            try:
                result = future.result()
            except Exception as e:
                print(f"[Multi-Query] search failed for a query variant: {e}")
                continue
            if result["best_score"] > best_score:
                best_score = result["best_score"]
            for doc, meta, score in zip(result["documents"], result["metadatas"], result["scores"]):
                key = re.sub(r"\s+", "", doc)[:200]
                if key not in chunk_map or score > chunk_map[key][0]:
                    chunk_map[key] = (score, doc, meta)
 
    if not chunk_map or best_score < SIMILARITY_THRESHOLD:
        return {"has_answer": False, "documents": [], "metadatas": [], "best_score": best_score}
 
    ranked = sorted(chunk_map.values(), key=lambda x: x[0], reverse=True)[:max_chunks]
 
    merged_docs, merged_meta, running_tokens = [], [], 0
    for score, doc, meta in ranked:
        doc_tokens = estimate_tokens(doc)
        if running_tokens + doc_tokens > max_context_tokens and merged_docs:
            break
        merged_docs.append(doc)
        merged_meta.append(meta)
        running_tokens += doc_tokens
 
    print(f"[Multi-Query] {len(chunk_map)} unique chunks found, kept {len(merged_docs)} "
          f"(~{running_tokens} estimated tokens, budget={max_context_tokens})")
    return {"has_answer": True, "documents": merged_docs, "metadatas": merged_meta, "best_score": best_score}

In [17]:
import random

# Cheap deterministic pre-filter: catches the vast majority of greetings/
# thanks/filler without spending any LLM call at all. Only genuinely
# ambiguous messages fall through to the LLM classifier below.
_CONVERSATIONAL_PATTERNS = [
    "شكرا", "شكراً", "تمام", "ماشي", "مرحبا", "أهلا", "اهلا", "يسلمو",
    "حسنا", "حسناً", "ممتاز", "السلام عليكم", "وعليكم السلام",
    "كيف حالك", "شو اخبارك", "مع السلامة", "باي", "طيب",
]

def _quick_conversational_check(user_message: str) -> bool | None:
    """Returns True/False if confidently classifiable without an LLM call,
    None if genuinely ambiguous and needs the LLM fallback."""
    stripped = user_message.strip()
    if len(stripped) <= 3:
        return True  # extremely short messages are almost always filler
    if any(p in stripped for p in _CONVERSATIONAL_PATTERNS) and len(stripped) < 25:
        return True
    if "؟" in stripped or "?" in stripped or len(stripped) > 25:
        return False  # a question mark or longer message is very likely a real question
    return None  # ambiguous — defer to LLM

def is_conversational(user_message: str) -> tuple[bool, str | None]:
    """
    Returns (True, polite_response) if message is a greeting/thanks/non-question.
    Returns (False, None) if message is a real academic question.
    """
    quick = _quick_conversational_check(user_message)
    if quick is False:
        return False, None
    if quick is True:
        # Still return a plausible generic Arabic reply — cheap, no LLM call
        return True, "أهلاً بك! كيف يمكنني مساعدتك؟"

    messages = [
        {"role": "system", "content": """You are a classifier for an academic advisor chatbot.
Decide if the student's message is a real academic question that needs retrieval, or just conversational
(greeting, thanks, acknowledgement, filler, or off-topic small talk).

Return ONLY JSON in this exact format:
{
  "is_conversational": true or false,
  "response": "a short polite Arabic reply if is_conversational is true, otherwise null"
}

Examples of conversational (is_conversational: true):
- "شكراً", "تمام", "ماشي", "مرحبا", "أهلاً", "يسلمو", "حسناً", "ممتاز", "السلام عليكم"
- "كيف حالك", "من أنت", "ما اسمك"
- Any greeting, farewell, or expression of thanks

Examples of real questions (is_conversational: false):
- "ما هي المنح الدراسية", "ما خطة الدراسة", "ما شرط القبول"
- "هل أنا مقبول بمعدل 75%", "ما المساقات في السنة الأولى"
- Any question about the program, courses, admission, or career

Rules:
- Return only JSON, no markdown, no explanation.
- The response field must always be in Arabic.
- Keep the response short (1-2 sentences), warm, and relevant to what the student said."""},
        {"role": "user", "content": user_message}
    ]

    def _try(client, model):
        resp = client.chat.completions.create(
            model=model, messages=messages, temperature=0.0, max_tokens=100
        )
        raw = re.sub(r"```json|```", "", resp.choices[0].message.content).strip()
        return json.loads(raw)

    try:
        result = _try(groq_client, "llama-3.1-8b-instant")
    except Exception as e:
        print(f"[is_conversational/Groq error] {e} — falling back to gpt-4o-mini")
        try:
            result = _try(openrouter_client, "openai/gpt-4o-mini")
        except Exception as e2:
            print(f"[is_conversational/fallback error] {e2} — treating as real question")
            return False, None   # ← this is the line that actually prevents the crash

    if result.get("is_conversational"):
        return True, result.get("response")
    return False, None

### 14. GRADIO CHAT

In [18]:
_CONDENSE_SYSTEM = """You are a query-rewriting component in an academic advising
system. Your ONLY job is to determine whether the student's latest message is a
complete, self-contained question, and if not, rewrite it into one using the
conversation history. You do NOT answer the question. You do NOT add facts that
aren't already present in the history or the message itself.

═══════════════════════════════════════════════════════════════════
STEP 1 — DECIDE: does this message depend on prior context to make sense?
═══════════════════════════════════════════════════════════════════
A message is SELF-CONTAINED (leave UNCHANGED) if a person with NO access to the
conversation history could understand exactly what is being asked — the subject
(program/course/policy), and any condition (a GPA, a track, a comparison target),
are all named explicitly in the message itself.

A message is DEPENDENT (must be rewritten) if understanding it requires knowing
something from an earlier turn — a pronoun, an omitted subject, an omitted
condition, or an implied continuation of a prior comparison/scenario.

═══════════════════════════════════════════════════════════════════
STEP 2 — IF DEPENDENT: identify which category of dependency this is, then repair it
═══════════════════════════════════════════════════════════════════

CATEGORY A — Pronoun / demonstrative reference
  Markers: "هذا التخصص", "هذا البرنامج", "هذا المساق", "هذه المنحة", "نفسه"
  Fix: replace the pronoun with the specific name it refers to, found by
  scanning backward through history for the most recently discussed program/
  course/policy. If more than one candidate could match, pick the one most
  recently and most centrally discussed (not just mentioned in passing).

CATEGORY B — Elliptical continuation ("و..." / "وماذا عن" / "ماذا عن")
  Markers: message starts with "و" attached to a bare noun phrase, or "ماذا عن"،
  with no verb or condition of its own — it's a sentence fragment, not a
  complete question.
  Fix: take the FULL question structure (verb, condition, question type) from
  the most recent prior question, and substitute only the new subject the
  fragment introduces. Everything else about the prior question's structure
  is preserved as-is.

CATEGORY C — Carried-over hypothetical condition
  Markers: the prior turn described a hypothetical (a GPA number, a track, a
  scenario) that does NOT belong to the actual student, and the new message
  continues discussing "that same scenario" without restating the condition.
  Fix: explicitly restate the hypothetical condition from the prior turn
  in the rewritten question, word-for-word as it was originally stated.
  CRITICAL: never substitute the actual student's own profile data (their
  real GPA, track, interests) for a hypothetical condition someone raised
  about a DIFFERENT, unnamed student. A hypothetical "طالب معدله 60" is not
  the current student — keep it hypothetical in the rewrite.

CATEGORY D — Comparative extension ("وبالمقارنة مع" / implied "what about the other one")
  Markers: the new message adds one more item to a comparison already
  established in a prior turn, without restating what's being compared.
  Fix: name both the new item and the original comparison subject(s)
  explicitly, and preserve what ASPECT is being compared (admission, courses,
  career paths, etc.) from the earlier turn.

CATEGORY E — Topic shift disguised as continuation
  Markers: the new message uses a pronoun or short form, but is actually
  about something with NO clear referent in recent history (the conversation
  moved on, or the reference is genuinely new).
  Fix: do NOT force a resolution. If no confident referent exists in the
  last few turns, return the message UNCHANGED rather than guessing — a
  wrong guess is worse than leaving it ambiguous for the next stage to handle.

═══════════════════════════════════════════════════════════════════
HARD CONSTRAINTS — apply to every rewrite
═══════════════════════════════════════════════════════════════════
1. Output ONLY the rewritten question. No explanation, no labels, no markdown,
   no quotes around it.
2. Keep the same language as the input (Arabic stays Arabic).
3. Do NOT answer the question — you are rewriting it, not responding to it.
4. Do NOT introduce any fact, number, program name, or condition that isn't
   explicitly present somewhere in the provided history or the message itself.
5. Do NOT merge unrelated subjects from different, distant turns just because
   they both appeared somewhere in history — only resolve against the most
   recent relevant turn(s).
6. If genuinely unsure whether a rewrite is needed or what it should be,
   prefer returning the message UNCHANGED over guessing.

═══════════════════════════════════════════════════════════════════
CRITICAL — THE EXAMPLES BELOW ARE ILLUSTRATIVE ONLY
═══════════════════════════════════════════════════════════════════
The examples use placeholder names like "تخصص أ" and "مساق ب" — these are
NOT real programs and NEVER appear in the actual conversation you're given.
Do NOT copy, adapt, or reuse any wording from these examples in your output,
even if the real conversation happens to resemble one of them closely.
Your output must be derived ONLY from the actual history and message you are
given in this specific call — never from this instructions text.

═══════════════════════════════════════════════════════════════════
EXAMPLES (placeholder names only — never mirror real program names)
═══════════════════════════════════════════════════════════════════

[Category A — pronoun reference]
History: user asked about "مساق أ" course content.
New: "من يدرّس هذا المساق؟"
Rewritten: "من يدرّس مساق أ؟"

[Category B — elliptical continuation]
History: user: "ما هي مساقات تخصص أ في السنة الأولى؟"
New: "وفي السنة الثانية؟"
Rewritten: "ما هي مساقات تخصص أ في السنة الثانية؟"

[Category C — hypothetical condition carryover]
History: user: "هل يمكن لطالب معدله أقل من 60 دخول تخصص أ؟"
         assistant: (eligibility answer)
New: "وتخصص ب؟"
Rewritten: "هل يمكن لطالب معدله أقل من 60 دخول تخصص ب؟"
(NOT the current student's own GPA — the hypothetical "أقل من 60" must be preserved.)

[Category D — comparative extension]
History: user: "ما الفرق من ناحية شروط القبول بين تخصص أ وتخصص ب؟"
New: "وتخصص ج؟"
Rewritten: "ما هو شرط القبول لتخصص ج، مقارنة بتخصص أ وتخصص ب؟"

[Category E — no confident referent, leave unchanged]
History: last 3 turns were all about scholarships, no program discussed recently.
New: "هل هذا يتطلب مقابلة شخصية؟"
Rewritten: "هل هذا يتطلب مقابلة شخصية؟"  (unchanged — "هذا" has no clear referent; do not guess)

[Already self-contained — no rewrite needed]
New: "من يدرّس مساق تراكيب بيانات؟"
Rewritten: "من يدرّس مساق تراكيب بيانات؟"  (unchanged — already names the subject explicitly)

CRITICAL FAILURE MODE TO AVOID:
Your output must NEVER still contain an unresolved reference word ("وماذا عن",
"هذا", "هذه", "نفسه") — if your output still has one of these, you have
failed to do your job. A correct rewrite REPLACES these words with the
actual subject; it never just returns the fragment as-is or restates a
DIFFERENT unresolved fragment.

CRITICAL — YOU ARE NOT A CONVERSATIONAL ASSISTANT:
=====================================================
You NEVER ask the student a clarifying question yourself. You NEVER request
additional information from them. You NEVER produce sub-questions about their
interests, GPA, or preferences unless the ORIGINAL message already asked
about exactly that. Your only two valid outputs are: (1) the original
message unchanged, or (2) the same question with its ambiguous reference
replaced by an explicit subject. Nothing else is a valid output, ever.
"""

def _low_lexical_overlap(original: str, rewritten: str, threshold: float = 0.2) -> bool:
    """
    A correct condensation PRESERVES most of the original question's wording,
    substituting only the unresolved pronoun/ellipsis. If the rewritten text
    shares almost no content words with the original, the model likely
    drifted into generating a DIFFERENT question entirely (e.g. asking the
    student a clarifying question, or answering) rather than resolving the
    one it was given.
    """
    def content_words(s: str) -> set:
        return set(w for w in re.findall(r"[\w\u0600-\u06FF]+", s) if len(w) > 2)

    orig = content_words(original)
    new = content_words(rewritten)
    if not orig:
        return False
    overlap = orig & new
    return (len(overlap) / len(orig)) < threshold

def condense_followup_question(user_message: str, memory: ConversationBufferWindowMemory) -> str:
    """
    Resolve elliptical follow-ups into standalone questions BEFORE retrieval
    or generation ever see them. This is the single point where pronoun
    resolution, ellipsis, and carried-over hypothetical conditions get
    resolved — replacing several scattered downstream patches.
    """
    chat_history = memory.load_memory_variables({})["history"]
    if not chat_history:
        return user_message

    history_msgs = [
        {"role": ("user" if m.type == "human" else "assistant"), "content": m.content}
        for m in chat_history[-4:]
    ]
    try:
        resp = github_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": _CONDENSE_SYSTEM},
                *history_msgs,
                {"role": "user", "content": user_message}
            ],
            temperature=0.0,
            max_tokens=150
        )
        rewritten = resp.choices[0].message.content.strip()
        if (rewritten
            and not _low_lexical_overlap(user_message, rewritten)):
          print(f"[Condense] '{user_message}' → '{rewritten}'")
          return rewritten

        print(f"[Condense] rejected (failed validation) — original: '{user_message}' | got: '{rewritten}'")
        return user_message
    except Exception as e:
        print(f"[Condense error] {e}")
        return user_message   # fail safe — proceed with the raw message

In [19]:
_MEMORY_STRIP_SYSTEM = """You are given an academic advisor's answer to a student.
Your task: return ONLY the factual/informational part of the answer — remove
any section that personalizes the answer to the specific student (GPA checks,
"يناسبك"/"يتوافق مع اهتماماتك" framing, eligibility verdicts, program
recommendations, "الاختيار الأمثل" or similar closing recommendations).

Rules:
- If the answer has NO personalization content at all, return it unchanged.
- Do NOT summarize or paraphrase the factual content — keep it word-for-word.
- Only remove personalization/recommendation content, nothing else.
- Return ONLY the trimmed answer text, no explanation, no markdown fences.
"""

def strip_personalization_tail(response: str) -> str:
    """
    Before saving an assistant turn to memory, remove personalization/
    recommendation content so a fit-related answer doesn't bias the tone
    of unrelated future turns still inside the k-window. The full
    personalized answer is still shown to the student this turn — only
    what gets remembered is trimmed.
    """
    if not response or len(response) < 50:
        return response
    try:
        resp = groq_client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[
                {"role": "system", "content": _MEMORY_STRIP_SYSTEM},
                {"role": "user", "content": response}
            ],
            temperature=0.0,
            max_tokens=len(response) // 2 + 200
        )
        trimmed = resp.choices[0].message.content.strip()
        if trimmed:
            print(f"[Memory] stripped {len(response) - len(trimmed)} chars of personalization")
            return trimmed
        return response
    except Exception as e:
        print(f"[Memory strip error] {e}")
        return response



In [20]:
import gradio as gr
from groq import APIStatusError

def get_empty_student() -> dict:
    return {"name": None, "email": None, "phone": None}

def get_empty_memory() -> ConversationBufferWindowMemory:
    return ConversationBufferWindowMemory(k=5, return_messages=True, memory_key="history")

def get_empty_profile() -> StudentProfile:
    return StudentProfile()

K = 3
def save_to_memory(memory: ConversationBufferWindowMemory, user_msg: str, assistant_msg: str):
    """Save a turn and manually enforce the k window."""
    memory.save_context({"input": user_msg}, {"output": assistant_msg})
    if len(memory.chat_memory.messages) > K * 2:
        memory.chat_memory.messages = memory.chat_memory.messages[-(K * 2):]

def extract_student_info(user_message: str, student: dict) -> dict:
    """Try to extract name/email/phone from the student's message."""
    try:
        resp = openrouter_client.chat.completions.create(
            model="openai/gpt-4o-mini",
            messages=[
                {"role": "system", "content": (
                    "Extract student contact details from the message if present."
                    "Respond ONLY in JSON: {'name': '...', 'email': '...', 'phone': '...'}"
                    "Use null for missing fields."
                )},
                {"role": "user", "content": user_message}
            ],
            temperature=0.0
        )
        extracted = json.loads(resp.choices[0].message.content)
        for key in ("name", "email", "phone"):
            if extracted.get(key):
                student[key] = extracted[key]
    except Exception:
        pass
    return student

def missing_contact(student: dict) -> str | None:
    if not student["name"]:
        return "name"
    if not student["email"] and not student["phone"]:
        return "contact"
    return None

def _append_user_turn(history: list, msg: str) -> None:
    history.append({"role": "user",      "content": msg})
    history.append({"role": "assistant", "content": ""})

def _set_last_assistant(history: list, text: str) -> None:
    if history and history[-1].get("role") == "assistant":
        history[-1]["content"] = text or ""

def _find_original_question(history: list, fallback: str) -> str:
    """Walk history backwards to find the user question that triggered a fallback reply."""
    msgs = history[:-1] if history else []
    for i in range(len(msgs) - 1, 0, -1):
        if msgs[i].get("role") == "assistant" and "لا تتوفر" in (msgs[i].get("content") or ""):
            prev = msgs[i - 1]
            if prev.get("role") == "user":
                return str(prev["content"])
    return fallback

import threading

def _save_to_memory_background(memory, user_message, response):
    """Runs after the response is already shown to the student — memory
    hygiene (personalization stripping + save) doesn't need to block the
    turn, since it only affects what future turns see, not this one."""
    try:
        cleaned = strip_personalization_tail(response)
        save_to_memory(memory, user_message, cleaned)
    except Exception as e:
        print(f"[Background memory save error] {e}")
        save_to_memory(memory, user_message, response)  # fail safe: save unstripped rather than lose the turn


def gradio_chat(
    user_message: str,
    history: list,
    student: dict,
    awaiting_info: bool,
    memory: ConversationBufferWindowMemory,
    profile: StudentProfile,
    onboarding: bool,
    pending_question: str | None,
):
    if not user_message.strip():
        return "", history, student, awaiting_info, memory, profile, onboarding, pending_question

    _append_user_turn(history, user_message)

    # ── Phase 1: Onboarding — collect student profile ──────────────────────
    if onboarding:
        profile  = extract_profile(user_message, profile)
        next_q   = next_onboarding_question(profile)

        if next_q:
            response = next_q
        else:
            onboarding = False
            response   = (
                "شكراً! الآن لديّ صورة واضحة عن وضعك الأكاديمي وأهدافك. "
                "يمكنني مساعدتك بشكل أفضل. ما سؤالك؟"
            )

        _set_last_assistant(history, response)
        return "", history, student, awaiting_info, memory, profile, onboarding, pending_question

    # ── Phase 2: Info-collection mode (post-fallback) ─────────────────────
    if awaiting_info:
        student       = extract_student_info(user_message, student)
        still_missing = missing_contact(student)

        if still_missing == "name":
            response = "شكراً! ما اسمك الكريم؟"
        elif still_missing == "contact":
            response = "شكراً! هل يمكنك تزويدي ببريدك الإلكتروني أو رقم هاتفك حتى يتمكن المرشد من التواصل معك؟"
        else:
            # original_q = _find_original_question(history, user_message)
            record_unknown_question(
                question=pending_question, name=student["name"],
                email=student.get("email"), phone=student.get("phone")
            )
            response      = "شكراً! تم إحالة سؤالك إلى المرشد الأكاديمي وسيتواصل معك قريباً. هل لديك أي سؤال آخر؟"
            awaiting_info = False
            pending_question = None

        _set_last_assistant(history, response)
        return "", history, student, awaiting_info, memory, profile, onboarding, pending_question

     # ── Conversational check — before any retrieval ────────────────────────
    conversational, polite_response = is_conversational(user_message)
    if conversational:
        _set_last_assistant(history, polite_response)
        return "", history, student, awaiting_info, memory, profile, onboarding, pending_question

    # ── Phase 3: Normal RAG flow ───────────────────────────────────────────
    resolved_message = condense_followup_question(user_message, memory)
    search_result = multi_query_search(resolved_message, memory, profile)

    # Layer 1: below threshold
    if not search_result["has_answer"]:
        print(f"[Layer 1] score={search_result['best_score']:.4f} — no relevant chunks")
        still_missing = missing_contact(student)
        if still_missing:
            awaiting_info = True
            pending_question = resolved_message
            response = (
                "لا تتوفر لديّ معلومات كافية للإجابة على هذا السؤال.\n"
                + ("سأحيل سؤالك إلى المرشد الأكاديمي. ما اسمك الكريم؟"
                   if still_missing == "name"
                   else "سأحيل سؤالك إلى المرشد الأكاديمي. هل يمكنك تزويدي ببريدك الإلكتروني أو رقم هاتفك؟")
            )
        else:
            record_unknown_question(
                question=resolved_message, name=student["name"],
                email=student.get("email"), phone=student.get("phone")
            )
            response = "لا تتوفر لديّ معلومات كافية للإجابة على هذا السؤال. تم إحالة سؤالك إلى المرشد الأكاديمي وسيتواصل معك قريباً."

        _set_last_assistant(history, response)
        return "", history, student, awaiting_info, memory, profile, onboarding, pending_question

    # Layer 2: pass chunks + profile to LLM
    print(f"[Layer 2] score={search_result['best_score']:.4f} — passing to LLM")
    context = build_context(search_result["documents"])

    chat_history = memory.load_memory_variables({})["history"]
    print(f"[MEMORY] {len(chat_history)} messages stored")
    history_msgs = [
        {"role": ("user" if m.type == "human" else "assistant"), "content": m.content}
        for m in chat_history
    ]

    # ── Force coverage of every program actually retrieved ─────────────────
    # Conversation history can anchor the model to a narrower comparison from
    # a previous turn. Derive the required program list from search_result
    # itself (ground truth) rather than trusting the model to infer scope
    # from phrasing + history.
    programs_in_context = sorted({
        m.get("program") for m in search_result["metadatas"] if m.get("program") #<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
    })

    coverage_instruction = ""
    if len(programs_in_context) > 2:
        coverage_instruction = (
            f"\n\nملاحظة إلزامية: السياق أعلاه يغطي {len(programs_in_context)} تخصصات بالتحديد: "
            f"{'، '.join(programs_in_context)}.\n"
            f"يجب أن تتناول إجابتك كل تخصص من هذه القائمة على حدة (حتى لو ذكرت المحادثة السابقة "
            f"تخصصات أقل) — لا تقتصر على التخصصات التي نوقشت سابقاً أو التي تبدو أكثر ملاءمة "
            f"لاهتمامات الطالب. ضع أي توصية شخصية مبنية على ملف الطالب في فقرة منفصلة وواضحة "
            f"في نهاية الإجابة فقط، بعد تغطية جميع التخصصات."
        )

    profile_ctx = profile.to_context_string()
    prompts = [
        {"role": "system", "content": SYSTEM_PROMPT},
        *history_msgs,
        {"role": "user", "content": (
            f"معلومات الطالب: {profile_ctx}\n"
            f"السياق:\n{context}\n"
            f"السؤال: {resolved_message}\n"
            f"Use student information only if the question explicitly concerns its relevance to them."
            f"(e.g., Is it suitable for me? Can I enroll? Is my GPA sufficient?) or other relative questions"
            f"Otherwise, answer the question as is, without analyzing eligibility or suitability.\n"
            f"{coverage_instruction}\n"
        )}
    ]

    response = ""
    done = False
    while not done:
        try:
            # resp_obj      = github_client.chat.completions.create(
            #     model="gpt-4o-mini", messages=prompts, tools=tools, temperature=0.1
            # )
            resp_obj = groq_client.chat.completions.create(
            model="qwen/qwen3-32b",
            messages=prompts,
            tools=tools,
            temperature=0.1
        )

        except APIStatusError as e:
            print(f"[Groq error] {e} — falling back to gpt-4o-mini")
            resp_obj = openrouter_client.chat.completions.create(
                model="openai/gpt-4o-mini",
                messages=prompts,
                tools=tools,
                temperature=0.1
            )
            
        finish_reason = resp_obj.choices[0].finish_reason
        message       = resp_obj.choices[0].message

        if finish_reason == "tool_calls":
            print("[Layer 2] LLM decided context insufficient — checking student contact")
            still_missing = missing_contact(student)
            if still_missing:
                awaiting_info = True
                pending_question = resolved_message
                response = (
                    "لا تتوفر لديّ معلومات كافية للإجابة على هذا السؤال.\n"
                    + ("سأحيل سؤالك إلى المرشد الأكاديمي. ما اسمك الكريم؟"
                       if still_missing == "name"
                       else "سأحيل سؤالك إلى المرشد الأكاديمي. هل يمكنك تزويدي ببريدك الإلكتروني أو رقم هاتفك؟")
                )
            else:
                record_unknown_question(
                    question=resolved_message,
                    name=student["name"],
                    email=student.get("email"),
                    phone=student.get("phone")
                )
                response = "لا تتوفر لديّ معلومات كافية للإجابة على هذا السؤال. تم إحالة سؤالك إلى المرشد الأكاديمي وسيتواصل معك قريباً."
                # handle_tool_calls(message.tool_calls)
                # response = "لا تتوفر لديّ معلومات كافية للإجابة على هذا السؤال. تم إحالة سؤالك إلى المرشد الأكاديمي وسيتواصل معك قريباً."
            done = True
        else:
            response = message.content
            done     = True

    if not awaiting_info:
        save_to_memory(memory, user_message, strip_personalization_tail(response))

    _set_last_assistant(history, response)
    return "", history, student, awaiting_info, memory, profile, onboarding, pending_question


# ── GRADIO UI ──────────────────────────────────────────────────────────────
def build_ui():
    with gr.Blocks(title="المستشار الأكاديمي الذكي — UCAS") as demo:
        gr.Markdown("""
        # المستشار الأكاديمي الذكي
        ### الكلية الجامعية للعلوم التطبيقية — تخصص علم البيانات والذكاء الاصطناعي
        ---
        """)

        student_state       = gr.State(get_empty_student)
        awaiting_info_state = gr.State(False)
        memory_state        = gr.State(get_empty_memory)
        profile_state       = gr.State(get_empty_profile)
        onboarding_state    = gr.State(True)
        pending_q_state     = gr.State(None)

        chatbot = gr.Chatbot(label="المحادثة", height=500, rtl=True)

        with gr.Row():
            msg_box  = gr.Textbox(placeholder="اكتب سؤالك هنا...", label="رسالتك", scale=9, rtl=True)
            send_btn = gr.Button("إرسال", scale=1, variant="primary")

        clear_btn = gr.Button("محادثة جديدة", variant="secondary")

        gr.Markdown("""
        > **ملاحظة:** هذا النظام يجيب فقط على أسئلة تخصص علم البيانات والذكاء الاصطناعي في UCAS.
        > إذا لم يتمكن من الإجابة، سيتم إحالة سؤالك تلقائياً إلى المرشد الأكاديمي.
        """)

        GREET = [{"role": "assistant", "content": (
            "مرحباً! أنا المستشار الأكاديمي الذكي في UCAS.\n"
            "قبل أن نبدأ، أودّ معرفة بعض المعلومات عنك لأتمكن من مساعدتك بشكل أفضل.\n"
            "للبدء، ما معدلك في الثانوية العامة (التوجيهي)؟ \n"
            "هذا يساعدني في معرفة البرامج التي تؤهل للقبول.\n"
        )}]

        io = [msg_box, chatbot, student_state, awaiting_info_state,
              memory_state, profile_state, onboarding_state, pending_q_state]

        send_btn.click(fn=gradio_chat, inputs=io, outputs=io)
        msg_box.submit(fn=gradio_chat,  inputs=io, outputs=io)

        clear_btn.click(
            fn=lambda: ("", [], get_empty_student(), False,
                        get_empty_memory(), get_empty_profile(), True, None),
            outputs=io
        ).then(fn=lambda: GREET, outputs=[chatbot])

        demo.load(fn=lambda: GREET, outputs=[chatbot])

    return demo

In [21]:
if __name__ == "__main__":
    DB_PATH = './chroma_db'
    client     = chromadb.PersistentClient(path=DB_PATH)
    collection = client.get_collection("ucas_knowledge_base")
    demo = build_ui()
    demo.launch(share=True, theme=gr.themes.Soft())

C:\Users\hp\AppData\Local\Temp\ipykernel_8768\64913762.py:8: LangChainDeprecationWarning: The class `ConversationBufferWindowMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  return ConversationBufferWindowMemory(k=5, return_messages=True, memory_key="history")


* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


[detect_metadata_filter raw] {"programs":["علم البيانات والذكاء الاصطناعي"],"course_codes":null,"course_names":null,"is_study_plan":false,"is_program_info":false,"is_scholarship":false,"is_career_opportunities":true,"all_programs":false}
[Filter] programs=['علم البيانات والذكاء الاصطناعي']
[Filter] course_codes=[]
[Filter] course_names=[]
  is_study_plan=False
  is_program_info=False
  is_scholarship=False
  is_career_opportunities=True
{'$and': [{'category': 'career_opportunities'}, {'program': 'علم البيانات والذكاء الاصطناعي'}]}
['التخصص: علم البيانات والذكاء الاصطناعي\nفرص العمل - المهام والمجالات الوظيفية العامة:\nتصميم البرامج الذكية وبناء خوارزميات لإنشاء تطبيقات تعمل على الذكاء الاصطناعي | إدارة النظم الذكية والمواقع الإلكترونية | العمل في الشركات والمؤسسات المتخصصة في التنقيب عن البيانات واسترجاع المعلومات | العمل عن بعد في مجال AI و Data Science | العمل في مجالات التفاعل بين الإنسان والحاسوب، الروبوتات، التعرف على الأشكال، الأنظمة الخبيرة', 'التخصص: علم البيانات والذكاء الاصطن